In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional
import time
from tqdm import tqdm
import os
import argparse
import h5py
import random
import torch.nn.functional as F

In [ ]:
class TrajectoryDataset(Dataset):
    """Dataset for neural ODE training from HDF5 files."""

    def __init__(self,
                 hdf5_file: str,
                 operator_type: str,
                 split: str = 'train'):
        """
        Initialize trajectory dataset from HDF5 file.
        
        Args:
            hdf5_file: Path to HDF5 file
            operator_type: 'heat' or 'dispersion'
            split: 'train', 'valid', or 'test'
        """
        self.hdf5_file = hdf5_file
        self.operator_type = operator_type
        self.split = split

        # Load data from HDF5 file
        with h5py.File(hdf5_file, 'r') as f:
            print(f"Loading {operator_type} data from {hdf5_file}")
            print(f"Available groups: {list(f.keys())}")

            # Load trajectories - assuming structure similar to train_combined.py
            if split in f:
                group = f[split]
                if 'pde_250-256' in group:
                    self.trajectories = group['pde_250-256'][:]  # (n_samples, n_timesteps, n_spatial)
                    self.alpha = group['alpha'][:]
                    self.beta = group['beta'][:]
                    self.gamma = group['gamma'][:]
                else:
                    raise ValueError(f"Expected 'pde_250-256' not found in group {split}")
            else:
                raise ValueError(f"Split '{split}' not found in file {hdf5_file}")

        self.n_samples, self.n_timesteps, self.n_spatial = self.trajectories.shape
        print(f"Loaded {self.n_samples} trajectories with shape ({self.n_timesteps}, {self.n_spatial})")

        # Create time points (assuming uniform spacing)
        self.time_points = np.linspace(0, 4.0, self.n_timesteps)
        self.dt = self.time_points[1] - self.time_points[0]

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        # Get appropriate parameter based on operator type
        if self.operator_type == 'heat':
            param_value = self.gamma[idx]  # gamma is typically the diffusion coefficient
        elif self.operator_type == 'dispersion':
            param_value = self.beta[idx]   # beta is typically the advection coefficient
        else:
            param_value = self.alpha[idx]  # fallback

        return {
            'u_sequence': torch.from_numpy(self.trajectories[idx]).float(),
            't_sequence': torch.from_numpy(self.time_points).float(),
            'parameter': torch.tensor(param_value).float(),
            'traj_idx': idx}

In [ ]:
def local_average_conv(u_trajectory, window=2):
    """
    Simple local averaging using F.conv1d with uniform weights
    
    Args:
        u_trajectory: (T, H) numpy array 
        window: averaging window size (3, 5, 7, etc.)
    """
    
    # Convert to tensor
    if isinstance(u_trajectory, np.ndarray):
        u_tensor = torch.from_numpy(u_trajectory).float()
        was_numpy = True
    else:
        u_tensor = u_trajectory.float()
        was_numpy = False
    
    T, H = u_tensor.shape
    
    # Create uniform averaging kernel
    kernel = torch.ones(1, 1, window) / window  # [1, 1, window]
    
    # Reshape for conv1d: (T, 1, H)
    u_reshaped = u_tensor.unsqueeze(1)  # (T, 1, H)
    
    # Circular padding for periodic BC
    pad = window // 2
    u_padded = F.pad(u_reshaped, (pad, pad), mode='circular')
    
    # Apply convolution
    u_smooth = F.conv1d(u_padded, kernel)
    
    # Remove channel dimension: (T, H)
    u_smooth = u_smooth.squeeze(1)
    
    if was_numpy:
        return u_smooth.numpy()
    return u_smooth

In [ ]:
#dataset = TrajectoryDataset("/mnt/home/lserrano/MP-Neural-PDE-Solvers/data/E_EULER_train_8192.h5", "euler")
dataset = TrajectoryDataset("/mnt/home/lserrano/LPSDA/data/E_EULER_train_8192.h5", "euler")

In [ ]:
traj_id=5099
u_trajectory = dataset[traj_id]["u_sequence"].numpy()   # shape: (T, N)
#u_trajectory = local_average_conv(u_trajectory, 5)
T, N = u_seq.shape
dx = 16/256

In [ ]:
# Assume u_trajectory has shape (T, H)
# dx is spatial spacing, dt is time step

# 1. Mass conservation (periodic BC)
mass = np.sum(u_trajectory, axis=1) * dx
print(f"Mass change: {abs(mass[-1] - mass[0]):.2e}")

# 2. Energy should decrease
energy = 0.5 * np.sum(u_trajectory**2, axis=1) * dx
energy_increases = np.sum(np.diff(energy) > 0)
print(f"Energy increases: {energy_increases} (should be 0)")
print(f"Energy dissipated: {energy[0] - energy[-1]:.3e}")

# 3. Total variation should not increase
tv = np.sum(np.abs(np.diff(u_trajectory, axis=1)), axis=1)
tv_increases = np.sum(np.diff(tv) > 0)
print(f"TV increases: {tv_increases} (should be 0)")

# 4. Maximum principle
u_min_init, u_max_init = np.min(u_trajectory[0]), np.max(u_trajectory[0])
u_min_all, u_max_all = np.min(u_trajectory), np.max(u_trajectory)
print(f"Range violation: {max(0, u_min_init - u_min_all, u_max_all - u_max_init):.2e}")

# Quick plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Energy evolution
ax1.plot(energy)
ax1.set_title('Energy (should decrease)')
ax1.grid(True)

# Final solution
ax2.plot(u_trajectory[-1])
ax2.set_title('Final solution')
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
for t in range(0, u_trajectory.shape[0], 25):
    plt.plot(u_trajectory[t])

In [ ]:
# Plot spectrum evolution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Spectrum over time
for t in [T//16, T//8, T//4, T//3]:
    u_fft = np.fft.fft(u_trajectory[t])
    k = np.fft.fftfreq(len(u_fft))
    spectrum = np.abs(u_fft)**2
    ax1.loglog(k[1:len(k)//2], spectrum[1:len(k)//2], label=f't={t}')

ax1.set_xlabel('Wavenumber k')
ax1.set_ylabel('E(k)')
ax1.legend()
ax1.set_title('Spectrum evolution')

# Look for k^-2 scaling at late times

In [ ]:
u_trajectory.shape